<a href="https://www.kaggle.com/code/zn1350/chess-game-training?scriptVersionId=334011602" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 在 Kaggle GPU 上训练小型评估网络

本 notebook 用于在 Kaggle Kernel（GPU）上做最小量的离线训练：
- 使用当前项目的 `ChessGame` 与 `AIEngine._evaluate` 作为标签来源（快速生成训练样本）
- 训练一个小型 CNN 估值网络（board -> scalar）
- 保存训练好的权重到 `model.pth`，供线上推理使用。

说明：这是轻量演示性脚本，目标是尽可能少的训练时间内得到可用权重。

In [1]:
# [保留原项目结构（推荐，适配原有代码）
# 若 Kaggle 环境缺少 torch，可取消下面注释安装（通常 Kaggle 已预装）
# !pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118
import os, sys, random, time

# 明确追加Kaggle工作目录到Python路径，确保能找到自定义模块
WORK_DIR = '/kaggle/input/models/zn1350/chess-game/pytorch/default/1'
sys.path.append(WORK_DIR)
os.chdir(WORK_DIR) # 切换到工作目录，保证相对路径一致

# 导入项目中的棋盘与评估函数（确保chess_game.py、ai_engine.py在工作目录下）
from chess_game import (
    ChessGame, R_KING, R_GUARD, R_BISHOP, R_ROOK, R_KNIGHT, R_CANNON, R_PAWN,
    B_KING, B_GUARD, B_BISHOP, B_ROOK, B_KNIGHT, B_CANNON, B_PAWN, EMPTY
)
from ai_engine import AIEngine

In [2]:
# 若 Kaggle 环境缺少 torch，可取消下面注释安装（通常 Kaggle 已预装）
# !pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118

In [3]:
import os, sys, random, time
# 确保工作目录为仓库根（Kaggle 上通常把 repo 放到 /kaggle/working）
sys.path.append('.')
os.getcwd()

'/kaggle/input/models/zn1350/chess-game/pytorch/default/1'

In [4]:
# 导入项目中的棋盘与评估函数
from chess_game import (
    ChessGame, R_KING, R_GUARD, R_BISHOP, R_ROOK, R_KNIGHT, R_CANNON, R_PAWN,
    B_KING, B_GUARD, B_BISHOP, B_ROOK, B_KNIGHT, B_CANNON, B_PAWN, EMPTY
)
from ai_engine import AIEngine
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [5]:
# 简单的棋盘编码：14 通道 (7 piece types * 2 colors) x 10 x 9
PIECE_LIST = [
    R_KING,
    R_GUARD,
    R_BISHOP,
    R_ROOK,
    R_KNIGHT,
    R_CANNON,
    R_PAWN,
    B_KING,
    B_GUARD,
    B_BISHOP,
    B_ROOK,
    B_KNIGHT,
    B_CANNON,
    B_PAWN,
]
PIECE_TO_IDX = {p: i for i, p in enumerate(PIECE_LIST)}

def board_to_tensor(board):
    # board: list of 10 rows x 9 cols with int piece ids
    x = np.zeros((14, 10, 9), dtype=np.float32)
    for r in range(10):
        for c in range(9):
            p = board[r][c]
            if p != 0:
                idx = PIECE_TO_IDX.get(p)
                if idx is not None:
                    x[idx, r, c] = 1.0
    return x

In [6]:
# 生成合成训练数据：用随机走子若干步，然后用当前 AI 的评估作为标签（离线标注）
def generate_dataset(n_positions=500, max_rand_moves=8, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    ai = AIEngine()
    X = []
    y = []
    for i in range(n_positions):
        g = ChessGame()
        moves = random.randint(0, max_rand_moves)
        for _ in range(moves):
            allm = g.get_all_moves(g.current_turn)
            if not allm:
                break
            mv = random.choice(allm)
            g.make_move(*mv)
        X.append(board_to_tensor(g.board))
        # 用 AI 内部评估函数作为标签（浮点）
        val = ai._evaluate(g)
        y.append(float(val))
    X = np.stack(X, axis=0)
    y = np.array(y, dtype=np.float32)
    return X, y

# 生成示例数据（可在 Kaggle 上调小/调大）
X, y = generate_dataset(n_positions=800, max_rand_moves=10)
print('X', X.shape, 'y', y.shape)

X (800, 14, 10, 9) y (800,)


In [7]:
class BoardDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(BoardDataset(X, y), batch_size=32, shuffle=True)

In [8]:
# 小型 CNN 估值网络
class SmallEvalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(14, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallEvalNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [9]:
# 训练循环（短期训练示例）
epochs = 6
for ep in range(epochs):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
    print(f'Epoch {ep+1}/{epochs} loss={total_loss/len(train_loader.dataset):.4f}')

Epoch 1/6 loss=54678.2311
Epoch 2/6 loss=52434.1763
Epoch 3/6 loss=42313.5591
Epoch 4/6 loss=36260.4569
Epoch 5/6 loss=35973.5920
Epoch 6/6 loss=35801.4221


In [10]:
# 保存模型到工作目录，Kaggle 可下载 artifacts
os.makedirs('/kaggle/working', exist_ok=True)
save_path = '/kaggle/working/model.pth'
torch.save({'state_dict': model.state_dict()}, save_path)
print('Saved model to', save_path)

Saved model to /kaggle/working/model.pth


## 接下来在项目中使用训练好的模型
在 `ai_engine.py` 的 `_evaluate` 中加载 `model.pth` 并用 `model.forward(board_tensor)` 得到估值（替换或与原评估混合）。建议先在 CPU 上做推理速度测试，再部署到实时系统。